# AssistiveMind v2 — Qwen QLoRA fine-tune (Kaggle)

**Before running:**
1. Notebook Settings (right sidebar) → **Accelerator: GPU T4 x2** (or P100) → **Internet: On**.
2. Create a Kaggle Dataset from the AssistiveMind v2 pack (upload
   `assistivemind_v2_core_chat.jsonl`, or the whole pack) and **Add Data**
   it to this notebook.
3. Run All.

Everything below is the same schema-normalization and training logic as
`training/format_data.py` / `training/train.py` / `training/validate_dataset.py`
in the Cup repo — pasted in directly so this notebook has no dependency on
GitHub repo access from Kaggle's side.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U "transformers==5.15.1" "trl==1.10.0" "peft==0.20.0" "accelerate==1.14.0" "datasets==5.0.1" "bitsandbytes>=0.43"

## Write out the training scripts (verbatim from the Cup repo)

In [ ]:
%%writefile format_data.py
"""Dataset schema normalization for AssistiveMind fine-tuning.

The dataset pack ships two schemas:
  - flat:  {"system": ..., "domain": ..., "instruction": ..., "response": ...}
  - chat:  {"domain": ..., "messages": [{"role": ..., "content": ...}, ...]}

Both are normalized to a `messages` list here, then formatted with the
target model's own chat template. That keeps the system prompt a real
training turn instead of metadata bolted onto a hand-built string, and
it means the training script never depends on an `output` key that
neither schema has ever had.
"""

USER_PREFIX = "User: "


def to_messages(example: dict) -> list[dict]:
    if "messages" in example:
        return example["messages"]

    if "instruction" in example and "response" in example:
        messages = []
        system = example.get("system")
        if system:
            messages.append({"role": "system", "content": system})

        instruction = example["instruction"]
        if instruction.startswith(USER_PREFIX):
            instruction = instruction[len(USER_PREFIX):]
        messages.append({"role": "user", "content": instruction})
        messages.append({"role": "assistant", "content": example["response"]})
        return messages

    raise KeyError(
        "Example matches neither the chat schema ('messages') nor the flat "
        f"schema ('instruction'/'response'). Keys found: {sorted(example.keys())}"
    )


def format_example(example: dict, tokenizer) -> str:
    messages = to_messages(example)
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )


In [ ]:
%%writefile train.py
#!/usr/bin/env python3
"""LoRA fine-tune Qwen on the AssistiveMind dataset.

Both dataset schemas (flat instruction/response and chat messages) are
supported transparently via format_data.to_messages — everything is run
through the model's own chat template, so the system prompt trains as a
real turn. Run training/validate_dataset.py on your data file first;
this script assumes it already passed.

Example (single 4090, 4-bit QLoRA):
    python training/train.py \\
        --dataset data/assistivemind_v2_core_chat.jsonl \\
        --model Qwen/Qwen2.5-7B-Instruct \\
        --output-dir runs/assistivemind-v2
"""
import argparse
import sys
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import SFTConfig, SFTTrainer

sys.path.insert(0, str(Path(__file__).resolve().parent))
from format_data import format_example

LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]


def parse_args():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="Qwen/Qwen2.5-7B-Instruct")
    ap.add_argument("--dataset", type=Path, required=True)
    ap.add_argument("--eval-dataset", type=Path, default=None)
    ap.add_argument("--output-dir", type=Path, default=Path("runs/assistivemind"))
    ap.add_argument("--epochs", type=float, default=3.0)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--batch-size", type=int, default=1)
    ap.add_argument("--grad-accum", type=int, default=8)
    ap.add_argument("--max-length", type=int, default=2048)
    ap.add_argument("--lora-r", type=int, default=16)
    ap.add_argument("--lora-alpha", type=int, default=32)
    ap.add_argument("--lora-dropout", type=float, default=0.05)
    ap.add_argument("--no-4bit", action="store_true", help="Disable 4-bit quantization")
    ap.add_argument("--seed", type=int, default=518)
    return ap.parse_args()


def load_model_and_tokenizer(args):
    tokenizer = AutoTokenizer.from_pretrained(args.model)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    quant_config = None
    if not args.no_4bit:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

    model = AutoModelForCausalLM.from_pretrained(
        args.model,
        quantization_config=quant_config,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

    if not args.no_4bit:
        model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        target_modules=LORA_TARGET_MODULES,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    return model, tokenizer


def build_dataset(path, tokenizer):
    ds = load_dataset("json", data_files=str(path), split="train")
    return ds.map(
        lambda example: {"text": format_example(example, tokenizer)},
        remove_columns=ds.column_names,
    )


def main():
    args = parse_args()

    model, tokenizer = load_model_and_tokenizer(args)

    train_ds = build_dataset(args.dataset, tokenizer)
    eval_ds = build_dataset(args.eval_dataset, tokenizer) if args.eval_dataset else None

    sft_config = SFTConfig(
        output_dir=str(args.output_dir),
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        max_length=args.max_length,
        dataset_text_field="text",
        # TRL 1.10 defaults to loss_type="chunked_nll", which patches the
        # model's forward for memory savings. That patch assumes forward is
        # a plain bound method, but device_map="auto" + 4-bit quantization
        # wraps it in a functools.partial via accelerate's hooks, crashing
        # with AttributeError: 'functools.partial' object has no attribute
        # '__func__'. Plain "nll" avoids the patch entirely.
        loss_type="nll",
        bf16=True,
        logging_steps=10,
        save_strategy="epoch",
        eval_strategy="epoch" if eval_ds is not None else "no",
        report_to="none",
        seed=args.seed,
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=tokenizer,
    )

    trainer.train()
    trainer.save_model(str(args.output_dir / "final"))
    tokenizer.save_pretrained(str(args.output_dir / "final"))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile validate_dataset.py
#!/usr/bin/env python3
"""Sanity-check a dataset file before spending GPU time on it.

Confirms every row converts to a valid `messages` list under the current
schema-normalization logic (see format_data.py), so a schema mismatch
shows up in seconds locally instead of after a training job has already
started on rented hardware.

Usage:
    python training/validate_dataset.py data/assistivemind_v2_core_chat.jsonl
"""
import argparse
import json
import sys
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
from format_data import to_messages


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("dataset", type=Path)
    args = ap.parse_args()

    schema_counts = Counter()
    domain_counts = Counter()
    errors = []

    with open(args.dataset, encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                example = json.loads(line)
            except json.JSONDecodeError as e:
                errors.append(f"line {line_no}: invalid JSON ({e})")
                continue

            domain_counts[example.get("domain", "<none>")] += 1

            if {"prompt", "expected", "avoid"} <= example.keys():
                # Behavioral-grading schema (e.g. assistivemind_v2_eval.jsonl):
                # a held-out prompt plus criteria to check a model's reply
                # against, not a supervised training pair. Not run through
                # to_messages/format_example.
                schema_counts["eval-criteria"] += 1
                continue

            schema_counts["chat" if "messages" in example else "flat"] += 1

            try:
                messages = to_messages(example)
            except KeyError as e:
                errors.append(f"line {line_no}: {e}")
                continue

            if not messages or messages[-1].get("role") != "assistant":
                last_role = messages[-1].get("role") if messages else None
                errors.append(
                    f"line {line_no}: last message role is {last_role!r}, expected 'assistant'"
                )

    total = sum(schema_counts.values())
    print(f"{args.dataset}: {total} examples")
    print(f"  schema mix: {dict(schema_counts)}")
    print(f"  domains: {dict(domain_counts)}")

    if errors:
        print(f"\n{len(errors)} problem(s) found:")
        for err in errors:
            print(f"  - {err}")
        raise SystemExit(1)

    print("\nAll examples converted cleanly.")


if __name__ == "__main__":
    main()


## Locate the dataset

Auto-detects the file regardless of what you named the Kaggle Dataset.

In [ ]:
import glob

candidates = glob.glob("/kaggle/input/**/assistivemind_v2_core_chat.jsonl", recursive=True)
assert candidates, (
    "Couldn't find assistivemind_v2_core_chat.jsonl under /kaggle/input. "
    "Upload it as a Kaggle Dataset and Add Data it to this notebook first."
)
DATA_PATH = candidates[0]
print("Using dataset:", DATA_PATH)

## Validate the dataset

Catches schema mismatches in seconds, before spending GPU time.

In [ ]:
!python validate_dataset.py "$DATA_PATH"

## Train

Sized for a 16GB Kaggle GPU (T4/P100): Qwen2.5-3B-Instruct in 4-bit QLoRA,
`max-length 768` (the dataset's longest example is ~1,370 characters total
across all messages, so this leaves comfortable headroom).

In [ ]:
!python train.py \
    --dataset "$DATA_PATH" \
    --model Qwen/Qwen2.5-3B-Instruct \
    --max-length 768 \
    --batch-size 2 \
    --grad-accum 4 \
    --epochs 3 \
    --output-dir /kaggle/working/runs/assistivemind-v2

## Package the adapter for download

Kaggle preserves `/kaggle/working` as notebook output when you Save & Run All / commit.

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/assistivemind-v2-lora",
    "zip",
    "/kaggle/working/runs/assistivemind-v2/final",
)
print("Wrote /kaggle/working/assistivemind-v2-lora.zip")